# Imports

In [2]:
import kagglehub
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import itertools
import gc
import pandas as pd

import torch
import torch.nn as nn

from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import timm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

seed=8359
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

/Users/ciccionos/Desktop/Secondo Primo Anno/Computer Vision/Project 2/cv-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Globals

In [3]:
KAGGLE_DATASET_NAME="ciccionoss/rrdataset-split-55-10-35"

MODEL_NAME="vit_base_patch16_224"
HIDDEN_DIM = 256
DROPOUT = 0.3
ACTIVATION_NAME = "silu"

HR_LEARNING_RATES = [1e-4, 3e-5]
HR_WEIGHT_DECAYS = [1e-4, 1e-3]
HR_LAMBDA_TRANSFORMS = [0.1, 0.25, 0.5, 0.75, 1.0, 2.0]
HR_BATCH_SIZES = [32]
HR_NUM_EPOCHS = 10
HR_PATIENCE = 5

LEARNING_RATE = 3e-5
WEIGHT_DECAY = 1e-3
LAMBDA_TRANSFORM = 0.25
BATCH_SIZE = 32
NUM_EPOCHS = 20
PATIENCE = 7

NUM_WORKERS = 0
NUM_WORKERS_TEST = 2

MODEL_SAVE_PATH=os.path.join(os.getcwd(), "models")

In [4]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Device: {device}")

Device: mps


# Utils

In [5]:
class CustomDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.root = root
        self.samples = []

        real_ai_map = {
            "real": 0,
            "ai": 1
        }

        transformation_map = {
            "original": 0,
            "redigital": 1,
            "transfer": 2
        }

        for transform_type in os.listdir(root):
            transform_path = os.path.join(root, transform_type)

            if not os.path.isdir(transform_path):
                continue

            for class_type in os.listdir(transform_path):
                class_path = os.path.join(transform_path, class_type)

                if not os.path.isdir(class_path):
                    continue

                for image_name in os.listdir(class_path):
                    image_path = os.path.join(class_path, image_name)

                    if transform_type not in transformation_map:
                        continue

                    if class_type not in real_ai_map:
                        continue

                    transformation_label = transformation_map[transform_type]
                    real_ai_label = real_ai_map[class_type]

                    self.samples.append(
                        (image_path, real_ai_label, transformation_label)
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, real_ai_label, transformation_label = self.samples[idx]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, real_ai_label, transformation_label

    def __str__(self):
        return f"This dataset has {len(self.samples)} samples"

In [6]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
def create_dataloaders(train_data, val_data, test_data, batch_size, dvc, num_work):
    pin_memory = True if dvc == "cuda" else False

    train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        dataset=val_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    return train_loader, val_loader, test_loader

# Data

In [7]:
print(f"Downloading {KAGGLE_DATASET_NAME} dataset!")
path = kagglehub.dataset_download(KAGGLE_DATASET_NAME)
print(f"Dataset downloaded at {path}!")

  0%|          | 11.0M/18.8G [00:04<2:11:33, 2.55MB/s]


KeyboardInterrupt: 

In [ ]:
path = os.path.join(os.getcwd, "\rrdataset-split-55-10-35\versions\1")

In [8]:
path = os.path.join(os.getcwd(), "archive")

In [9]:
dataset_train = CustomDataset(
    root=path + "/train",
    transform=data_transform
)

dataset_val = CustomDataset(
    root=path + "/val",
    transform=data_transform
)

dataset_test = CustomDataset(
    root=path + "/test",
    transform=data_transform
)
print(f"Dataset train: {dataset_train}, Dataset val: {dataset_val}, Dataset test: {dataset_test}")

Dataset train: This dataset has 2550 samples, Dataset val: This dataset has 2549 samples, Dataset test: This dataset has 45900 samples


In [10]:
real_ai_names = {
    0: "real",
    1: "ai"
}

transformation_names = {
    0: "original",
    1: "redigital",
    2: "transfer"
}

def dataset_balance_table(dataset):
    samples_df = pd.DataFrame(
        dataset.samples,
        columns=["image_path", "real_ai_label", "transformation_label"]
    )
    samples_df["real_ai"] = samples_df["real_ai_label"].map(real_ai_names)
    samples_df["transformation"] = samples_df["transformation_label"].map(transformation_names)

    balance_table = pd.crosstab(
        samples_df["transformation"],
        samples_df["real_ai"]
    ).reindex(
        index=list(transformation_names.values()),
        columns=list(real_ai_names.values()),
        fill_value=0
    )
    balance_table["total"] = balance_table.sum(axis=1)

    total_row = balance_table.sum(axis=0).to_frame().T
    total_row.index = ["total"]

    return pd.concat([balance_table, total_row])

datasets = {
    "train": dataset_train,
    "val": dataset_val,
    "test": dataset_test
}

balance_summary = []
balance_tables = {}

for split_name, dataset in datasets.items():
    balance_table = dataset_balance_table(dataset)
    joint_counts = balance_table.loc[
        list(transformation_names.values()),
        list(real_ai_names.values())
    ]
    min_joint_count = int(joint_counts.to_numpy().min())
    max_joint_count = int(joint_counts.to_numpy().max())
    max_joint_gap = max_joint_count - min_joint_count

    balance_summary.append({
        "split": split_name,
        "samples": len(dataset),
        "min_joint_count": min_joint_count,
        "max_joint_count": max_joint_count,
        "max_joint_gap": max_joint_gap,
        "balanced": max_joint_gap <= 1
    })
    balance_tables[split_name] = balance_table

balance_summary = pd.DataFrame(balance_summary)
display(balance_summary)

for split_name, balance_table in balance_tables.items():
    print(f"{split_name} joint distribution")
    display(balance_table)

,split,samples,min_joint_count,max_joint_count,max_joint_gap,balanced
0,train,2550,425,425,0,True
1,val,2549,424,425,1,True
2,test,45900,7650,7650,0,True


train joint distribution


real_ai,real,ai,total
original,425,425,850
redigital,425,425,850
transfer,425,425,850
total,1275,1275,2550


val joint distribution


real_ai,real,ai,total
original,425,425,850
redigital,424,425,849
transfer,425,425,850
total,1274,1275,2549


test joint distribution


real_ai,real,ai,total
original,7650,7650,15300
redigital,7650,7650,15300
transfer,7650,7650,15300
total,22950,22950,45900


# Network

In [10]:
class MultiHeadModel(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_224",
        num_transform_classes=3,
        hidden_dim=256,
        dropout=0.3,
        activation_name="gelu"
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        match activation_name:
            case "gelu":
                activation_real_ai = nn.GELU()
                activation_transform = nn.GELU()
            case "relu":
                activation_real_ai = nn.ReLU()
                activation_transform = nn.ReLU()
            case "leaky_relu":
                activation_real_ai = nn.LeakyReLU(0.01)
                activation_transform = nn.LeakyReLU(0.01)
            case "silu":
                activation_real_ai = nn.SiLU()
                activation_transform = nn.SiLU()
            case _:
                activation_real_ai = nn.GELU()
                activation_transform = nn.GELU()

        self.real_ai_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            activation_real_ai,
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

        self.transformation_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            activation_transform,
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_transform_classes)
        )

    def forward(self, x):
        features = self.backbone(x)

        real_ai_logits = self.real_ai_head(features).squeeze(1)
        transformation_logits = self.transformation_head(features)

        return real_ai_logits, transformation_logits

In [11]:
class SingleHeadBinaryModel(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_224",
        hidden_dim=256,
        dropout=0.3
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        self.real_ai_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        features = self.backbone(x)

        real_ai_logits = self.real_ai_head(features).squeeze(1)

        return real_ai_logits

In [12]:
class SingleHeadTransformartionModel(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_224",
        num_transform_classes=3,
        hidden_dim=256,
        dropout=0.3
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        self.transformation_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_transform_classes)
        )

    def forward(self, x):
        features = self.backbone(x)

        transformation_logits = self.transformation_head(features)

        return transformation_logits

# Train

In [13]:
def evaluate_model(model, data_loader, lambda_transform, device):
    model.eval()

    bce_loss = nn.BCEWithLogitsLoss()
    ce_loss = nn.CrossEntropyLoss()

    total_loss = 0
    total_samples = 0

    all_fake_labels = []
    all_fake_preds = []

    all_transform_labels = []
    all_transform_preds = []

    with torch.no_grad():
        for images, fake_labels, transform_labels in tqdm(
            data_loader,
            desc="Validation",
            disable=True
          ):
            images = images.to(device)
            fake_labels = fake_labels.to(device)
            transform_labels = transform_labels.to(device)

            fake_logits, transform_logits = model(images)

            loss_fake = bce_loss(fake_logits, fake_labels.float())
            loss_transform = ce_loss(transform_logits, transform_labels)

            loss = (1 - lambda_transform) * loss_fake + lambda_transform * loss_transform

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()
            transform_preds = torch.argmax(transform_logits, dim=1)

            all_fake_labels.extend(fake_labels.cpu().numpy())
            all_fake_preds.extend(fake_preds.cpu().numpy())

            all_transform_labels.extend(transform_labels.cpu().numpy())
            all_transform_preds.extend(transform_preds.cpu().numpy())

    avg_loss = total_loss / total_samples

    real_ai_acc = accuracy_score(all_fake_labels, all_fake_preds)
    real_ai_f1 = f1_score(
        all_fake_labels,
        all_fake_preds,
        zero_division=0
    )

    transform_acc = accuracy_score(
        all_transform_labels,
        all_transform_preds
    )
    transform_macro_f1 = f1_score(
        all_transform_labels,
        all_transform_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "real_ai_acc": real_ai_acc,
        "real_ai_f1": real_ai_f1,
        "transform_acc": transform_acc,
        "transform_macro_f1": transform_macro_f1
    }

In [14]:
def train_model(
    model_name,
    learning_rate,
    weight_decay,
    lambda_transform,
    batch_size,
    hidden_dim,
    dropout,
    activation_name,
    num_epochs=10,
    patience=3
):

  train_loader, val_loader, _ = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    batch_size,
    device,
    NUM_WORKERS
  )

  model = MultiHeadModel(
      model_name=model_name,
      num_transform_classes=3,
      hidden_dim=hidden_dim,
      dropout=dropout,
      activation_name=activation_name
  ).to(device)

  bce_loss = nn.BCEWithLogitsLoss()
  ce_loss = nn.CrossEntropyLoss()

  optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=learning_rate*0.01
  )

  history = []

  best_score=-1.0
  best_epoch=0
  best_val_metrics = None
  epochs_without_improvements=0

  model_file_name = (
      f"{model_name}_"
    + f"lr-{learning_rate}_"
    + f"wd-{weight_decay}_"
    + f"lt-{lambda_transform}_"
    + f"bs-{batch_size}_"
    + f"hd-{hidden_dim}_"
    + f"d-{dropout}_"
    + f"a-{activation_name}_"
    + f"seed-{seed}_MultiHead.pth"
  )

  if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
  model_path = os.path.join(MODEL_SAVE_PATH, model_file_name)

  for epoch in range(num_epochs):
    model.train()

    current_lr = optimizer.param_groups[0]["lr"]

    train_total_loss = 0.0
    train_total_samples = 0

    all_train_fake_labels = []
    all_train_fake_preds = []

    all_train_transform_labels = []
    all_train_transform_preds = []

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    for images, fake_labels, transform_labels in tqdm(
        train_loader,
        desc=(
            f"Train | LR={learning_rate} | WD={weight_decay} | "
            f"LT={lambda_transform} | BS={batch_size} | "
            f"Epoch {epoch + 1}/{num_epochs}"
        )
    ):
      images = images.to(device)

      fake_labels = fake_labels.to(device)
      transform_labels = transform_labels.to(device)

      fake_logits, transform_logits = model(images)

      loss_fake = bce_loss(fake_logits, fake_labels.float())
      loss_transform = ce_loss(transform_logits, transform_labels)

      loss = (1 - lambda_transform) * loss_fake + lambda_transform * loss_transform

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      batch_size_current = images.size(0)

      train_total_loss += loss.item() * batch_size_current
      train_total_samples += batch_size_current

      fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()
      transform_preds = torch.argmax(transform_logits, dim=1)

      all_train_fake_labels.extend(fake_labels.cpu().numpy())
      all_train_fake_preds.extend(fake_preds.cpu().numpy())

      all_train_transform_labels.extend(transform_labels.cpu().numpy())
      all_train_transform_preds.extend(transform_preds.cpu().numpy())

    avg_train_loss = train_total_loss / train_total_samples

    train_real_ai_acc = accuracy_score(
        all_train_fake_labels,
        all_train_fake_preds
    )
    train_real_ai_f1 = f1_score(
        all_train_fake_labels,
        all_train_fake_preds,
        zero_division=0
    )
    train_transform_acc = accuracy_score(
        all_train_transform_labels,
        all_train_transform_preds
    )
    train_transform_macro_f1 = f1_score(
        all_train_transform_labels,
        all_train_transform_preds,
        average="macro",
        zero_division=0
    )

    val_metrics = evaluate_model(
        model,
        val_loader,
        lambda_transform,
        device
    )

    scheduler.step()

    epoch_result = {
        "epoch": epoch + 1,
        "learning_rate": learning_rate,
        "current_lr": current_lr,
        "weight_decay": weight_decay,
        "lambda_transform": lambda_transform,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "dropout": dropout,
        "activation": activation_name,

        "train_loss": avg_train_loss,
        "train_real_ai_acc": train_real_ai_acc,
        "train_real_ai_f1": train_real_ai_f1,
        "train_transform_acc": train_transform_acc,
        "train_transform_macro_f1": train_transform_macro_f1,

        "val_loss": val_metrics["loss"],
        "val_real_ai_acc": val_metrics["real_ai_acc"],
        "val_real_ai_f1": val_metrics["real_ai_f1"],
        "val_transform_acc": val_metrics["transform_acc"],
        "val_transform_macro_f1": val_metrics["transform_macro_f1"]
    }

    history.append(epoch_result)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Real AI Acc: {train_real_ai_acc:.4f} | Train Real AI F1: {train_real_ai_f1:.4f} | "
        f"Train Transform Acc: {train_transform_acc:.4f} | Train Transform F1: {train_transform_macro_f1:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Real AI Acc: {val_metrics['real_ai_acc']:.4f} | Val Real AI F1: {val_metrics['real_ai_f1']:.4f} | "
        f"Val Transform Acc: {val_metrics['transform_acc']:.4f} | Val Transform F1: {val_metrics['transform_macro_f1']:.4f}"
    )

    current_score = (
      0.8 * val_metrics["real_ai_acc"]
      + 0.2 * val_metrics["transform_acc"]
    )

    if current_score > best_score:
      best_score = current_score
      best_epoch = epoch + 1
      best_val_metrics = val_metrics
      epochs_without_improvements = 0
      torch.save(
          model.state_dict(),
          model_path
      )
      print (f"Saved new best model: {model_path}!")
    else:
      epochs_without_improvements += 1

    if epochs_without_improvements >= patience:
      print(
        f"Early stopping at epoch {epoch + 1}! |"
        f"No improvements in {patience} epochs."
      )
      break
  best_result = {
    "best_epoch": best_epoch,
    "best_score": best_score,
    "best_val_metrics": best_val_metrics,
    "history": history
  }

  del model
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  return best_result

In [15]:
def evaluate_model_binary(model, data_loader, device):
    model.eval()

    bce_loss = nn.BCEWithLogitsLoss()

    total_loss = 0
    total_samples = 0

    all_fake_labels = []
    all_fake_preds = []

    with torch.no_grad():
        for images, fake_labels, _ in tqdm(
            data_loader,
            desc="Validation",
            disable=True
          ):
            images = images.to(device)
            fake_labels = fake_labels.to(device)

            fake_logits = model(images)

            loss_fake = bce_loss(fake_logits, fake_labels.float())

            batch_size = images.size(0)

            total_loss += loss_fake.item() * batch_size
            total_samples += batch_size

            fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()

            all_fake_labels.extend(fake_labels.cpu().numpy())
            all_fake_preds.extend(fake_preds.cpu().numpy())

    avg_loss = total_loss / total_samples

    real_ai_acc = accuracy_score(all_fake_labels, all_fake_preds)
    real_ai_f1 = f1_score(
        all_fake_labels,
        all_fake_preds,
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "real_ai_acc": real_ai_acc,
        "real_ai_f1": real_ai_f1
    }

In [16]:
def train_model_binary(
    model_name,
    learning_rate,
    weight_decay,
    batch_size,
    hidden_dim,
    dropout,
    num_epochs=10,
    patience=3
):

  train_loader, val_loader, _ = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    batch_size,
    device,
    NUM_WORKERS
  )

  model = SingleHeadBinaryModel(
      model_name=model_name,
      hidden_dim=hidden_dim,
      dropout=dropout
  ).to(device)

  bce_loss = nn.BCEWithLogitsLoss()

  optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=learning_rate*0.01
  )

  history = []

  best_score=-1.0
  best_epoch=0
  best_val_metrics = None
  epochs_without_improvements=0

  model_file_name = (
      f"{model_name}_"
    + f"lr-{learning_rate}_"
    + f"wd-{weight_decay}_"
    + f"bs-{batch_size}_"
    + f"hd-{hidden_dim}_"
    + f"d-{dropout}_"
    + f"seed-{seed}_BinaryHead.pth"
  )

  if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
  model_path = os.path.join(MODEL_SAVE_PATH, model_file_name)

  for epoch in range(num_epochs):
    model.train()

    current_lr = optimizer.param_groups[0]["lr"]

    train_total_loss = 0.0
    train_total_samples = 0

    all_train_fake_labels = []
    all_train_fake_preds = []

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    for images, fake_labels, _ in tqdm(
        train_loader,
        desc=(
            f"Train | LR={learning_rate} | WD={weight_decay} | "
            f"BS={batch_size} | Epoch {epoch + 1}/{num_epochs}"
        )
    ):
      images = images.to(device)

      fake_labels = fake_labels.to(device)

      fake_logits = model(images)

      loss_fake = bce_loss(fake_logits, fake_labels.float())

      optimizer.zero_grad()
      loss_fake.backward()
      optimizer.step()

      batch_size_current = images.size(0)

      train_total_loss += loss_fake.item() * batch_size_current
      train_total_samples += batch_size_current

      fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()

      all_train_fake_labels.extend(fake_labels.cpu().numpy())
      all_train_fake_preds.extend(fake_preds.cpu().numpy())

    avg_train_loss = train_total_loss / train_total_samples

    train_real_ai_acc = accuracy_score(
        all_train_fake_labels,
        all_train_fake_preds
    )
    train_real_ai_f1 = f1_score(
        all_train_fake_labels,
        all_train_fake_preds,
        zero_division=0
    )

    val_metrics = evaluate_model_binary(
        model,
        val_loader,
        device
    )

    scheduler.step()

    epoch_result = {
        "epoch": epoch + 1,
        "learning_rate": learning_rate,
        "current_lr": current_lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "dropout": dropout,

        "train_loss": avg_train_loss,
        "train_real_ai_acc": train_real_ai_acc,
        "train_real_ai_f1": train_real_ai_f1,

        "val_loss": val_metrics["loss"],
        "val_real_ai_acc": val_metrics["real_ai_acc"],
        "val_real_ai_f1": val_metrics["real_ai_f1"]
    }

    history.append(epoch_result)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_real_ai_acc:.4f} | Train F1: {train_real_ai_f1:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['real_ai_acc']:.4f} | Val F1: {val_metrics['real_ai_f1']:.4f}"
    )

    current_score = val_metrics["real_ai_acc"]

    if current_score > best_score:
      best_score = current_score
      best_epoch = epoch + 1
      best_val_metrics = val_metrics
      epochs_without_improvements = 0
      torch.save(
          model.state_dict(),
          model_path
      )
      print (f"Saved new best model: {model_path}!")
    else:
      epochs_without_improvements += 1

    if epochs_without_improvements >= patience:
      print(
        f"Early stopping at epoch {epoch + 1}! |"
        f"No improvements in {patience} epochs."
      )
      break
  best_result = {
    "best_epoch": best_epoch,
    "best_score": best_score,
    "best_val_metrics": best_val_metrics,
    "history": history
  }

  del model
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  return best_result

In [17]:
def evaluate_model_transformation(model, data_loader, device):
    model.eval()

    ce_loss = nn.CrossEntropyLoss()

    total_loss = 0
    total_samples = 0

    all_transform_labels = []
    all_transform_preds = []

    with torch.no_grad():
        for images, _, transform_labels in tqdm(
            data_loader,
            desc="Validation",
            disable=True
          ):
            images = images.to(device)
            transform_labels = transform_labels.to(device)

            transform_logits = model(images)

            loss_transform = ce_loss(transform_logits, transform_labels)

            batch_size = images.size(0)

            total_loss += loss_transform.item() * batch_size
            total_samples += batch_size

            transform_preds = torch.argmax(transform_logits, dim=1)

            all_transform_labels.extend(transform_labels.cpu().numpy())
            all_transform_preds.extend(transform_preds.cpu().numpy())

    avg_loss = total_loss / total_samples

    transform_acc = accuracy_score(
        all_transform_labels,
        all_transform_preds
    )
    transform_macro_f1 = f1_score(
        all_transform_labels,
        all_transform_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "transform_acc": transform_acc,
        "transform_macro_f1": transform_macro_f1
    }

In [18]:
def train_model_transformation(
    model_name,
    learning_rate,
    weight_decay,
    batch_size,
    hidden_dim,
    dropout,
    num_epochs=10,
    patience=3
):

  train_loader, val_loader, _ = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    batch_size,
    device,
    NUM_WORKERS
  )

  model = SingleHeadTransformartionModel(
      model_name=model_name,
      num_transform_classes=3,
      hidden_dim=hidden_dim,
      dropout=dropout
  ).to(device)

  ce_loss = nn.CrossEntropyLoss()

  optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=learning_rate*0.01
  )

  history = []

  best_score=-1.0
  best_epoch=0
  best_val_metrics = None
  epochs_without_improvements=0

  model_file_name = (
      f"{model_name}_"
    + f"lr-{learning_rate}_"
    + f"wd-{weight_decay}_"
    + f"bs-{batch_size}_"
    + f"hd-{hidden_dim}_"
    + f"d-{dropout}_"
    + f"seed-{seed}_TransformationHead.pth"
  )

  if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
  model_path = os.path.join(MODEL_SAVE_PATH, model_file_name)

  for epoch in range(num_epochs):
    model.train()

    current_lr = optimizer.param_groups[0]["lr"]

    train_total_loss = 0.0
    train_total_samples = 0

    all_train_transform_labels = []
    all_train_transform_preds = []

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    for images, _, transform_labels in tqdm(
        train_loader,
        desc=(
            f"Train | LR={learning_rate} | WD={weight_decay} | "
            f"BS={batch_size} | Epoch {epoch + 1}/{num_epochs}"
        )
    ):
      images = images.to(device)

      transform_labels = transform_labels.to(device)

      transform_logits = model(images)

      loss_transform = ce_loss(transform_logits, transform_labels)

      optimizer.zero_grad()
      loss_transform.backward()
      optimizer.step()

      batch_size_current = images.size(0)

      train_total_loss += loss_transform.item() * batch_size_current
      train_total_samples += batch_size_current

      transform_preds = torch.argmax(transform_logits, dim=1)

      all_train_transform_labels.extend(transform_labels.cpu().numpy())
      all_train_transform_preds.extend(transform_preds.cpu().numpy())

    avg_train_loss = train_total_loss / train_total_samples

    train_transform_acc = accuracy_score(
        all_train_transform_labels,
        all_train_transform_preds
    )
    train_transform_macro_f1 = f1_score(
        all_train_transform_labels,
        all_train_transform_preds,
        average="macro",
        zero_division=0
    )

    val_metrics = evaluate_model_transformation(
        model,
        val_loader,
        device
    )

    scheduler.step()

    epoch_result = {
        "epoch": epoch + 1,
        "learning_rate": learning_rate,
        "current_lr": current_lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "dropout": dropout,

        "train_loss": avg_train_loss,
        "train_transform_acc": train_transform_acc,
        "train_transform_macro_f1": train_transform_macro_f1,

        "val_loss": val_metrics["loss"],
        "val_transform_acc": val_metrics["transform_acc"],
        "val_transform_macro_f1": val_metrics["transform_macro_f1"]
    }

    history.append(epoch_result)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_transform_acc:.4f} | Train F1: {train_transform_macro_f1:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['transform_acc']:.4f} | Val F1: {val_metrics['transform_macro_f1']:.4f}"
    )

    current_score = val_metrics["transform_acc"]

    if current_score > best_score:
      best_score = current_score
      best_epoch = epoch + 1
      best_val_metrics = val_metrics
      epochs_without_improvements = 0
      torch.save(
          model.state_dict(),
          model_path
      )
      print (f"Saved new best model: {model_path}!")
    else:
      epochs_without_improvements += 1

    if epochs_without_improvements >= patience:
      print(
        f"Early stopping at epoch {epoch + 1}! |"
        f"No improvements in {patience} epochs."
      )
      break
  best_result = {
    "best_epoch": best_epoch,
    "best_score": best_score,
    "best_val_metrics": best_val_metrics,
    "history": history
  }

  del model
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  return best_result

With Hyper-Parameter Research

Without Hyper-Parameter Research

In [19]:
best_result_no_hr = train_model(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lambda_transform=LAMBDA_TRANSFORM,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    activation_name=ACTIVATION_NAME,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE
)

best_result_binary_no_hr = train_model_binary(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE
)

best_result_transformation_no_hr = train_model_transformation(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE
)


Epoch [1/20]


Train | LR=3e-05 | WD=0.001 | LT=0.25 | BS=32 | Epoch 1/20:  52%|█████▏    | 457/877 [05:49<05:00,  1.40it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (178562880 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | LT=0.25 | BS=32 | Epoch 1/20:  58%|█████▊    | 511/877 [06:32<05:05,  1.20it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (102960000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | LT=0.25 | BS=32 | Epoch 1/20: 100%|██████████| 877/877 [11:18<00:00,  1.29it/s]
c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (92160000 p

Epoch [1/20] - Train Loss: 0.2316 | Train Real AI Acc: 0.9336 | Train Real AI F1: 0.9334 | Train Transform Acc: 0.8158 | Train Transform F1: 0.8161 | Val Loss: 0.1265 | Val Real AI Acc: 0.9614 | Val Real AI F1: 0.9622 | Val Transform Acc: 0.9080 | Val Transform F1: 0.9076
Saved new best model: /content/models/vit_base_patch16_224_lr-3e-05_wd-0.001_lt-0.25_bs-32_hd-256_d-0.3_a-silu_seed-8359_MultiHead.pth!

Epoch [2/20]


Train | LR=3e-05 | WD=0.001 | LT=0.25 | BS=32 | Epoch 2/20:  77%|███████▋  | 672/877 [06:31<02:02,  1.68it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (102960000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | LT=0.25 | BS=32 | Epoch 2/20:  80%|███████▉  | 699/877 [06:49<01:48,  1.63it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (178562880 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | LT=0.25 | BS=32 | Epoch 2/20: 100%|██████████| 877/877 [08:32<00:00,  1.71it/s]
c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (92160000 p

Epoch [2/20] - Train Loss: 0.0779 | Train Real AI Acc: 0.9868 | Train Real AI F1: 0.9868 | Train Transform Acc: 0.9191 | Train Transform F1: 0.9192 | Val Loss: 0.1028 | Val Real AI Acc: 0.9731 | Val Real AI F1: 0.9733 | Val Transform Acc: 0.9124 | Val Transform F1: 0.9131
Saved new best model: /content/models/vit_base_patch16_224_lr-3e-05_wd-0.001_lt-0.25_bs-32_hd-256_d-0.3_a-silu_seed-8359_MultiHead.pth!

Epoch [3/20]


...

Epoch [8/20] - Train Loss: 0.0598 | Train Acc: 0.9752 | Train F1: 0.9752 | Val Loss: 0.2119 | Val Acc: 0.9127 | Val F1: 0.9126

Epoch [9/20]


Train | LR=3e-05 | WD=0.001 | BS=32 | Epoch 9/20:   3%|▎         | 28/877 [00:16<09:12,  1.54it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (102960000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | BS=32 | Epoch 9/20:  71%|███████   | 624/877 [06:00<02:38,  1.59it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (178562880 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | BS=32 | Epoch 9/20: 100%|██████████| 877/877 [08:31<00:00,  1.71it/s]
c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (92160000 pixels) exceeds limit of 8947848

Epoch [9/20] - Train Loss: 0.0524 | Train Acc: 0.9783 | Train F1: 0.9783 | Val Loss: 0.2254 | Val Acc: 0.9078 | Val F1: 0.9078

Epoch [10/20]


Train | LR=3e-05 | WD=0.001 | BS=32 | Epoch 10/20:  72%|███████▏  | 629/877 [06:07<02:31,  1.64it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (178562880 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | BS=32 | Epoch 10/20:  83%|████████▎ | 732/877 [07:06<01:18,  1.85it/s]c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (102960000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Train | LR=3e-05 | WD=0.001 | BS=32 | Epoch 10/20: 100%|██████████| 877/877 [08:30<00:00,  1.72it/s]
c:\Users\mirco\OneDrive\Desktop\Computer Vision Project\cv-project\.venv\Lib\site-packages\PIL\Image.py:3574: DecompressionBombWarning: Image size (92160000 pixels) exceeds limit of 894

Epoch [10/20] - Train Loss: 0.0432 | Train Acc: 0.9821 | Train F1: 0.9821 | Val Loss: 0.2659 | Val Acc: 0.9182 | Val F1: 0.9185
Early stopping at epoch 10! |No improvements in 7 epochs.


In [20]:
# Extract and save history for each model
history_multihead = best_result_no_hr["history"]
history_binary = best_result_binary_no_hr["history"]
history_transformation = best_result_transformation_no_hr["history"]

# Convert to DataFrames for easier analysis
df_history_multihead = pd.DataFrame(history_multihead)
df_history_binary = pd.DataFrame(history_binary)
df_history_transformation = pd.DataFrame(history_transformation)

print("MultiHead Model Training History:")
print(df_history_multihead.head())
print(f"\nShape: {df_history_multihead.shape}")

print("\n\nBinary Model Training History:")
print(df_history_binary.head())
print(f"\nShape: {df_history_binary.shape}")

print("\n\nTransformation Model Training History:")
print(df_history_transformation.head())
print(f"\nShape: {df_history_transformation.shape}")

# Optional: Save to CSV files
csv_save_path = os.path.join(os.getcwd(), "training_history")
if not os.path.exists(csv_save_path):
    os.makedirs(csv_save_path)

df_history_multihead.to_csv(os.path.join(csv_save_path, "history_multihead.csv"), index=False)
df_history_binary.to_csv(os.path.join(csv_save_path, "history_binary.csv"), index=False)
df_history_transformation.to_csv(os.path.join(csv_save_path, "history_transformation.csv"), index=False)

print(f"\n\nTraining histories saved to {csv_save_path}")

MultiHead Model Training History:
   epoch  learning_rate  current_lr  weight_decay  lambda_transform  \
0      1        0.00003    0.000030         0.001              0.25   
1      2        0.00003    0.000030         0.001              0.25   
2      3        0.00003    0.000029         0.001              0.25   
3      4        0.00003    0.000028         0.001              0.25   
4      5        0.00003    0.000027         0.001              0.25   

   batch_size  hidden_dim  dropout activation  train_loss  train_real_ai_acc  \
0          32         256      0.3       silu    0.231552           0.933581   
1          32         256      0.3       silu    0.077925           0.986844   
2          32         256      0.3       silu    0.054633           0.992834   
3          32         256      0.3       silu    0.041766           0.995187   
4          32         256      0.3       silu    0.037269           0.995579   

   train_real_ai_f1  train_transform_acc  train_transform_

# Testing

In [ ]:
binary_model_path = os.path.join(MODEL_SAVE_PATH, "models/vit_base_patch16_224_lr-3e-05_wd-0.001_bs-32_hd-256_d-0.3_seed-8359_BinaryHead.pth")

In [ ]:
loaded_model = torch.load(
    binary_model_path,
    map_location=torch.device(device)
)
model = SingleHeadBinaryModel()
model.load_state_dict(loaded_model['model_state_dict'])
model.to(device)
print("Binary model loaded!")

In [ ]:
model.eval()

all_real_ai_labels = []
all_real_ai_preds = []

print("Starting binary model testing...")

with torch.no_grad():
    for images, real_ai_labels, transformation_labels in tqdm(
        dataset_test,
        desc="Testing",
        total=len(dataset_test)
    ):
        images = images.to(device)
        real_ai_labels = real_ai_labels.to(device).float()

        real_ai_logits = model(images)

        real_ai_preds = (torch.sigmoid(real_ai_logits) > 0.5).float()

        all_real_ai_labels.extend(real_ai_labels.cpu().numpy())
        all_real_ai_preds.extend(real_ai_preds.cpu().numpy())

# Calculate metrics
real_ai_accuracy = accuracy_score(all_real_ai_labels, all_real_ai_preds)
real_ai_cm = confusion_matrix(all_real_ai_labels, all_real_ai_preds)

print(f"Testing Binary model complete!")
print(f"Accuracy: {real_ai_accuracy:.4f}")
print("\nConfusion Matrix:")
print(real_ai_cm)

In [ ]:
transformation_model_path = os.path.join(MODEL_SAVE_PATH, "models/vit_base_patch16_224_lr-3e-05_wd-0.001_bs-32_hd-256_d-0.3_seed-8359_TransformationHead.pth")

In [ ]:
loaded_model = torch.load(
    transformation_model_path,
    map_location=torch.device(device)
)
model = SingleHeadTransformartionModel()
model.load_state_dict(loaded_model['model_state_dict'])
model.to(device)
print("Transformation model loaded!")

In [ ]:
model.eval()

all_transformation_labels = []
all_transformation_preds = []

print("Starting model testing...")

with torch.no_grad():
    for images, real_ai_labels, transformation_labels in tqdm(
        dataset_test,
        desc="Testing",
        total=len(dataset_test)
    ):
        images = images.to(device)
        transformation_labels = transformation_labels.to(device)

        transformation_logits = model(images)

        transformation_preds = torch.argmax(transformation_logits, dim=1)

        all_transformation_labels.extend(transformation_labels.cpu().numpy())
        all_transformation_preds.extend(transformation_preds.cpu().numpy())

# Calculate metrics
transformation_accuracy = accuracy_score(all_transformation_labels, all_transformation_preds)
transformation_cm = confusion_matrix(all_transformation_labels, all_transformation_preds)

print(f"Testing Transformation model complete!")
print(f"Transformation Accuracy: {transformation_accuracy:.4f}")
print("\nTransformation Confusion Matrix:")
print(transformation_cm)

In [ ]:
double_head_model_path = os.path.join(MODEL_SAVE_PATH, "models/vit_base_patch16_224_lr-3e-05_wd-0.001_bs-32_hd-256_d-0.3_seed-8359_MultiHead.pth")

In [ ]:
loaded_model = torch.load(
    double_head_model_path,
    map_location=torch.device(device)
)
model = MultiHeadModel()
model.load_state_dict(loaded_model['model_state_dict'])
model.to(device)
print("Multi-Head model loaded!")

In [ ]:
model.eval()

all_real_ai_labels = []
all_real_ai_preds = []
all_transformation_labels = []
all_transformation_preds = []

print("Starting model testing...")

with torch.no_grad():
    for images, real_ai_labels, transformation_labels in tqdm(
        dataset_test,
        desc="Testing",
        total=len(dataset_test)
    ):
        images = images.to(device)
        real_ai_labels = real_ai_labels.to(device).float()
        transformation_labels = transformation_labels.to(device)

        real_ai_logits, transformation_logits = model(images)

        real_ai_preds = (torch.sigmoid(real_ai_logits) > 0.5).float()
        transformation_preds = torch.argmax(transformation_logits, dim=1)

        all_real_ai_labels.extend(real_ai_labels.cpu().numpy())
        all_real_ai_preds.extend(real_ai_preds.cpu().numpy())
        all_transformation_labels.extend(transformation_labels.cpu().numpy())
        all_transformation_preds.extend(transformation_preds.cpu().numpy())

# Calculate metrics
real_ai_accuracy = accuracy_score(all_real_ai_labels, all_real_ai_preds)
transformation_accuracy = accuracy_score(all_transformation_labels, all_transformation_preds)
real_ai_cm = confusion_matrix(all_real_ai_labels, all_real_ai_preds)
transformation_cm = confusion_matrix(all_transformation_labels, all_transformation_preds)

print(f"Testing complete!")
print(f"Real/AI Head Accuracy: {real_ai_accuracy:.4f}")
print(f"Transformation Head Accuracy: {transformation_accuracy:.4f}")
print("\nReal/AI Head Confusion Matrix:")
print(real_ai_cm)
print("\nTransformation Head Confusion Matrix:")
print(transformation_cm)